In [0]:
"""
id: source_0
template: source
templateVersion: 2.0.0
name: train_test_datasset
position:
  x: -31
  y: 89
description:
  text: Load data from a specified CSV file.
  hash: c90f50c7
previewCodeHash: 7d21718276acab73
previewMode: "1000"
config:
  file_source:
    path: /Volumes/datacartel_dbx/havg_data/volumen/train_test_datasset.csv
    format: '"csv"'
input: []
"""

# generated from the system
from typing import Dict, Any, List

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def _build_metric_view_sql(
    table_name: str, dims: List[str], measures: List[str]
) -> str:
    def q(n: str) -> str:
        return "`" + n.replace("`", "``") + "`"

    select_parts = [q(d) for d in dims] + [f"MEASURE({q(m)}) AS {q(m)}" for m in measures]
    if not select_parts:
        return f"SELECT * FROM {table_name}"
    sql = f"SELECT {', '.join(select_parts)} FROM {table_name}"
    if dims and measures:
        sql += " GROUP BY " + ", ".join(q(d) for d in dims)
    elif dims and not measures:
        sql = f"SELECT DISTINCT {', '.join(q(d) for d in dims)} FROM {table_name}"
    return sql

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")

        mv_selection = table_source.get("metricView")
        if mv_selection is not None:
            dims = mv_selection.get("dimensions")
            measures = mv_selection.get("measures")
            if dims is None or measures is None:
                raise ValueError(
                    "Source: metricView selection is incomplete (both "
                    "'dimensions' and 'measures' must be explicit lists). "
                    "Re-open the source node and select the metric view "
                    "again to re-seed the picker."
                )
            sql = _build_metric_view_sql(table_name, list(dims), list(measures))
            out = spark.sql(sql)
        elif table_source.get("isExpression"):
            out = spark.sql(table_name)
        else:
            out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        globals()["ld_display_outputs"] = str(
            dbutils.widgets.getAll().get("ld_display_outputs", "true")
        ).strip().lower() not in ("false", "0", "no", "off")
    except Exception:
        globals()["ld_display_outputs"] = True
ctx = globals().setdefault("ctx", {})
config = {
    "file_source": {
        "path": "/Volumes/datacartel_dbx/havg_data/volumen/train_test_datasset.csv",
        "format": "\"csv\""
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["source_0.data"] = out["data"]
if globals().get("ld_display_outputs", True):
    display(ctx["source_0.data"])

In [0]:
"""
id: python_2_1
template: python
templateVersion: 1.0.0
name: pyspark_ingest
position:
  x: 199
  y: 88.0625
description:
  text: Reload a module and prepare data for output.
  hash: 1df201f5
previewCodeHash: 3767409d8676c14b
previewMode: "1000"
config:
  code: |-
    import importlib
    import src.pyspark_train.ingest as ingest

    importlib.reload(ingest)

    from src.pyspark_train.ingest import run_ingest

    print(run_ingest.__code__.co_consts)
input:
  - node: source_0
    input_port: data
    output_port: data
"""

# generated from the system
from typing import Dict, Any
from pyspark.sql import DataFrame

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    data = inputs.get("data", [] if True else None)
    result = data[0] if data else spark.createDataFrame([], "col: string")

    import importlib
    import src.pyspark_train.ingest as ingest

    importlib.reload(ingest)

    from src.pyspark_train.ingest import run_ingest

    print(run_ingest.__code__.co_consts)

    return {"result": result}

# generated from the system
if "ld_display_outputs" not in globals():
    try:
        globals()["ld_display_outputs"] = str(
            dbutils.widgets.getAll().get("ld_display_outputs", "true")
        ).strip().lower() not in ("false", "0", "no", "off")
    except Exception:
        globals()["ld_display_outputs"] = True
ctx = globals().setdefault("ctx", {})
config = {}
inputs = {
    "data": [
        ctx["source_0.data"]
    ],
    "data__sources": [
        {
            "node": "source_0",
            "output_port": "data",
            "name": "train_test_datasset",
            "df_name": "train_test_datasset"
        }
    ]
}
out = run(config, inputs, spark)
ctx["python_2_1.result"] = out["result"]
if globals().get("ld_display_outputs", True):
    display(ctx["python_2_1.result"])